# 06 - NASA C-MAPSS RUL Regression Baselines
### Phase 2: Machine Learning Baseline Development & Evaluation

This notebook evaluates baseline regression models for Remaining Useful Life (RUL) estimation on the **NASA C-MAPSS FD001** benchmark dataset:
- Linear Regression
- Random Forest Regressor
- Gradient Boosting Regressor
- XGBoost Regressor

Evaluations strictly adhere to:
1. Engine-level training/validation split (no window leakage).
2. Official NASA test terminal cycle benchmark (one prediction per engine at final observed cycle).
3. NASA Asymmetric Scoring Function & RMSE/MAE evaluation.


In [ ]:
import sys
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Add src to sys.path
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(project_root / "src"))

from evaluation.metrics import compute_regression_metrics, nasa_asymmetric_scoring_function
from training.nasa_rul_trainer import load_nasa_tabular_data


## 1. Load Processed Tabular Data & Features


In [ ]:
proc_dir = project_root / "data" / "processed" / "nasa"
train_df, val_df, test_df, feature_cols = load_nasa_tabular_data("FD001", processed_dir=proc_dir)

print(f"Train Shape: {train_df.shape} ({train_df['unit_id'].nunique()} engines)")
print(f"Val Shape:   {val_df.shape} ({val_df['unit_id'].nunique()} engines)")
print(f"Test Shape:  {test_df.shape} ({test_df['unit_id'].nunique()} engines)")
print(f"Total Features: {len(feature_cols)}")


## 2. Load Saved Baseline Models & Evaluate on Validation Fleet


In [ ]:
models_dir = project_root / "models" / "baseline"
model_names = [
    "Linear Regression",
    "Random Forest Regressor",
    "Gradient Boosting Regressor",
    "XGBoost Regressor"
]

X_val = val_df[feature_cols].to_numpy(dtype=np.float32)
y_val = val_df["RUL"].to_numpy(dtype=np.float32)

val_summary = []
loaded_models = {}

for name in model_names:
    clean_name = name.lower().replace(" ", "_")
    model_path = models_dir / f"nasa_rul_{clean_name}_FD001.joblib"
    if model_path.exists():
        model = joblib.load(model_path)
        loaded_models[name] = model
        y_val_pred = model.predict(X_val)
        m = compute_regression_metrics(y_val, y_val_pred, prefix="val")
        val_summary.append({"Model": name, **m})

df_val_results = pd.DataFrame(val_summary)
display(df_val_results)


## 3. Official NASA Test Terminal Benchmark Evaluation


In [ ]:
# Extract terminal cycle per test engine
test_terminal_df = test_df.groupby("unit_id").last().reset_index()
X_test_term = test_terminal_df[feature_cols].to_numpy(dtype=np.float32)
y_test_term = test_terminal_df["RUL"].to_numpy(dtype=np.float32)

term_summary = []
for name, model in loaded_models.items():
    y_test_pred = model.predict(X_test_term)
    m = compute_regression_metrics(y_test_term, y_test_pred, prefix="test_terminal")
    term_summary.append({"Model": name, **m})

df_term_results = pd.DataFrame(term_summary)
display(df_term_results)


## 4. Predicted vs Actual RUL (Best Model)


In [ ]:
best_model_name = df_val_results.sort_values(by="val_rmse").iloc[0]["Model"]
print(f"Best Model Selected by Validation RMSE: {best_model_name}")

best_model = loaded_models[best_model_name]
y_pred_best = best_model.predict(X_test_term)

plt.figure(figsize=(7, 6))
plt.scatter(y_test_term, y_pred_best, alpha=0.75, color="#1f77b4", edgecolors="k", s=45)
min_v = min(y_test_term.min(), y_pred_best.min())
max_v = max(y_test_term.max(), y_pred_best.max())
plt.plot([min_v, max_v], [min_v, max_v], "r--", lw=2, label="Perfect Agreement (y = x)")
plt.title(f"Predicted vs Actual RUL on Terminal Test Fleet ({best_model_name})")
plt.xlabel("Actual Terminal RUL (Cycles)")
plt.ylabel("Predicted Terminal RUL (Cycles)")
plt.legend()
plt.tight_layout()
plt.show()
